# Notebook 2: Train Continual SD-LoRA Adapter

Bu notebook tek crop icin adapter egitir, OOD hazirligini olcer ve export ciktisini kaydeder.

Onerilen kullanim sirasi:
1. Calisma kimligi hucresinde `CROP_NAME` ve `PART_NAME` belirleyin.
2. Parametreleri duzenleyin.
3. Guncelleme/erisim kontrolu hucresini calistirin.
4. Hucreleri sirayla calistirin.
5. Is bitince `guided/00_start_here.md` ile ciktilari okuyun.


## Hizli Akis

Bu notebook secilen tek adapter icin training, OOD kalibrasyonu, final evaluation ve export akisini sirayla calistirir.

- Genelde sadece `ADAPTER_KEY` degistirin; crop, part, dataset, OOD ve OE yollari bu anahtardan acik sekilde set edilir.
- Runtime dataset beklentisi: `data/prepared_runtime_datasets/<adapter_key>/continual|val|test|ood|oe`.
- `ood` final kanit, `oe` sadece training yardimci negatifleridir; ikisini karistirmayin.
- Push veya publish son adimda bozulursa training bittiyse run klasorunu kurtarmayi deneyin, bastan egitime donmeyin.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from urllib.parse import urlsplit, urlunsplit

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
NOTEBOOK2_SPARSE_PATHS = (
    'README.md',
    'docs',
    'src',
    'scripts',
    'config',
    'colab_notebooks',
    'requirements.txt',
    'requirements_colab.txt',
    'pyproject.toml',
)

def _build_repo_access_url(repo_url, token):
    token = str(token or '').strip()
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _ensure_aads_repo_on_path():
    candidates = [Path.cwd(), Path.cwd().parent, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    clone_url = _build_repo_access_url(REPO_URL, os.environ.get('GH_TOKEN') or os.environ.get('GITHUB_TOKEN'))
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', clone_url, str(CLONE_TARGET)], check=True)
        subprocess.run(['git', 'sparse-checkout', 'set', *NOTEBOOK2_SPARSE_PATHS], cwd=str(CLONE_TARGET), check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET

_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell01_bootstrap_access.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell03_runtime_setup.py', globals())


In [ ]:
# Adapter secimi: genelde sadece bunu degistir.
ADAPTER_KEY = "grape__fruit"  # grape__fruit, grape__leaf, strawberry__fruit, strawberry__leaf, tomato__fruit, tomato__leaf

# Bayesian optimizasyon default olarak acik: onceki run'lardan ayni adapter icin parametre onerisi uygular.
ENABLE_BAYESIAN_OPTIMIZATION = True

# Adapter bazli gorunur defaultlar. ADAPTER_KEY hangi bloğun kullanilacagini secer.
ADAPTER_RECS = {
    "grape__fruit": {"crop": "grape", "part": "fruit", "ood": "data/prepared_runtime_datasets/grape__fruit/ood", "oe": "data/prepared_runtime_datasets/grape__fruit/oe", "oe_enabled": True, "oe_w": 0.20, "allow_under_min": False, "overrides": {"EPOCHS": 32, "BATCH_SIZE": 48, "LEARNING_RATE": 1e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.15, "OOD_FACTOR": 3.0}},
    "grape__leaf": {"crop": "grape", "part": "leaf", "ood": "data/prepared_runtime_datasets/grape__leaf/ood", "oe": "data/prepared_runtime_datasets/grape__leaf/oe", "oe_enabled": True, "oe_w": 0.20, "allow_under_min": False, "overrides": {"EPOCHS": 28, "BATCH_SIZE": 64, "LEARNING_RATE": 1e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.12, "OOD_FACTOR": 3.0}},
    "strawberry__fruit": {"crop": "strawberry", "part": "fruit", "ood": "data/prepared_runtime_datasets/strawberry__fruit/ood", "oe": "data/prepared_runtime_datasets/strawberry__fruit/oe", "oe_enabled": True, "oe_w": 0.10, "allow_under_min": True, "overrides": {"EPOCHS": 36, "BATCH_SIZE": 32, "LEARNING_RATE": 8e-5, "LORA_R": 16, "LORA_ALPHA": 16, "LORA_DROPOUT": 0.18, "OOD_FACTOR": 3.0}},
    "strawberry__leaf": {"crop": "strawberry", "part": "leaf", "ood": "data/prepared_runtime_datasets/strawberry__leaf/ood", "oe": "data/prepared_runtime_datasets/strawberry__leaf/oe", "oe_enabled": True, "oe_w": 0.15, "allow_under_min": False, "overrides": {"EPOCHS": 22, "BATCH_SIZE": 96, "LEARNING_RATE": 1.5e-4, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.10, "OOD_FACTOR": 3.0}},
    "tomato__fruit": {"crop": "tomato", "part": "fruit", "ood": "data/prepared_runtime_datasets/tomato__fruit/ood", "oe": "data/prepared_runtime_datasets/tomato__fruit/oe", "oe_enabled": True, "oe_w": 0.15, "allow_under_min": False, "overrides": {"EPOCHS": 30, "BATCH_SIZE": 64, "LEARNING_RATE": 8e-5, "LORA_R": 24, "LORA_ALPHA": 24, "LORA_DROPOUT": 0.15, "OOD_FACTOR": 3.0}},
    "tomato__leaf": {"crop": "tomato", "part": "leaf", "ood": "data/prepared_runtime_datasets/tomato__leaf/ood", "oe": "data/prepared_runtime_datasets/tomato__leaf/oe", "oe_enabled": True, "oe_w": 0.15, "allow_under_min": False, "overrides": {"EPOCHS": 20, "BATCH_SIZE": 96, "LEARNING_RATE": 1.2e-4, "LORA_R": 32, "LORA_ALPHA": 32, "LORA_DROPOUT": 0.10, "OOD_FACTOR": 3.0}},
}

# Adapterden bagimsiz gorunur defaultlar.
DEFAULT_RUNTIME_PARAMS = {
    "ENABLE_BAYESIAN_OPTIMIZATION": ENABLE_BAYESIAN_OPTIMIZATION,
    "WEIGHT_DECAY": 0.01,
    "LOSS_NAME": "logitnorm",
    "LOGITNORM_TAU": 1.0,
    "MIXED_PRECISION": "bf16",
    "GRAD_ACCUM_STEPS": 1,
    "MAX_GRAD_NORM": 1.0,
    "LABEL_SMOOTHING": 0.0,
    "SCHEDULER_NAME": "cosine",
    "SCHEDULER_WARMUP_RATIO": 0.1,
    "SCHEDULER_MIN_LR": 1e-6,
    "EARLY_STOPPING_PATIENCE": 6,
    "EARLY_STOPPING_MIN_DELTA": 0.0,
    "DETERMINISTIC": False,
    "SEED": 42,
    "RANDAUGMENT_NUM_OPS": 2,
    "RANDAUGMENT_MAGNITUDE": 7,
    "NUM_WORKERS": 12,
    "PREFETCH": 8,
    "PIN_MEMORY": True,
    "USE_CACHE": True,
    "CACHE_TRAIN_SPLIT": True,
    "VALIDATION_EVERY_N_EPOCHS": 1,
    "CHECKPOINT_EVERY_N_STEPS": 250,
    "CHECKPOINT_ON_EXCEPTION": True,
    "STDOUT_BATCH_INTERVAL": 12,
    "RESUME_MODE": "fresh",
    "AUTO_DISCONNECT_RUNTIME": True,
    "AUTO_DISCONNECT_GRACE_SECONDS": 20,
    "AUTO_PUSH_TO_GITHUB": True,
    "AUTO_PUSH_REMOTE_NAME": "origin",
    "AUTO_PUSH_BRANCH": None,
}

# Sadece bu run icin defaultlari ezmek istersen buraya yaz.
MANUAL_PARAM_OVERRIDES = {}

with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    from scripts.notebook_helpers.cell_script_runner import run_cell_script
    run_cell_script('nb2_cell04_parameter_resolution.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell05_access_check.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell06_dataset_validation.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell07_engine_init.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell08_ood_config_verify.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell09_training.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell10_ood_calibration.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell11_adapter_save.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb2_cell12_final_evaluation.py', globals())
